In [2]:
import pandas as pd
import duckdb as ddb

from pathlib import Path

In [3]:
word_list: list[str] = []
with Path("../data/word_list.txt").open() as f:
    word_list = f.read().split()
if not word_list:
    print("Failed to load word list")

In [4]:
char_tuples = [
    (word, *tuple(char for char in word))
    for word in word_list
]

In [7]:
words_df = pd.DataFrame.from_records(char_tuples, columns=["word", "l0", "l1", "l2", "l3", "l4"])
words_df

,word,l0,l1,l2,l3,l4
0,AAHED,A,A,H,E,D
1,AALII,A,A,L,I,I
2,AARGH,A,A,R,G,H
3,AARTI,A,A,R,T,I
4,ABACA,A,B,A,C,A
...,...,...,...,...,...,...
12915,ZUZIM,Z,U,Z,I,M
12916,ZYGAL,Z,Y,G,A,L
12917,ZYGON,Z,Y,G,O,N
12918,ZYMES,Z,Y,M,E,S


In [19]:
with ddb.connect("../data/words_db") as conn, Path("../data/schema.sql").open() as schema_file:
    schema_script = schema_file.read()
    conn.execute(schema_script)
    conn.executemany("INSERT INTO MyWordleOlap.MyWordList VALUES (?, ?, ?, ?, ?, ?)", char_tuples)

In [20]:
# verification
with ddb.connect("../data/words_db") as conn:
    qr = conn.execute("SELECT * FROM MyWordleOlap.MyWordList;").fetch_df()
qr

,word,l0,l1,l2,l3,l4
0,AAHED,A,A,H,E,D
1,AALII,A,A,L,I,I
2,AARGH,A,A,R,G,H
3,AARTI,A,A,R,T,I
4,ABACA,A,B,A,C,A
...,...,...,...,...,...,...
12915,ZUZIM,Z,U,Z,I,M
12916,ZYGAL,Z,Y,G,A,L
12917,ZYGON,Z,Y,G,O,N
12918,ZYMES,Z,Y,M,E,S
